# Vibrating drum modes

*waves · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/membrane_modes)

The normal modes of a square or circular drum skin. The 2D Laplacian on a grid is built from the 1D one with Kronecker products, $\Delta = D\otimes I + I\otimes D$, and a single symmetric eigensolver (eigh) returns every mode shape and frequency at once.

**Vector form**

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\nabla^2 u\quad\Rightarrow\quad -\nabla^2\phi = \frac{\omega^2}{c^2}\,\phi,\qquad u = \phi\,\cos\omega t$$

**Index (tensor) form**

$$\Delta_{(ab)(cd)} = D_{ac}\,\delta_{bd} + \delta_{ac}\,D_{bd},\qquad -\Delta_{pq}\,\phi_{qm} = \frac{\omega_m^2}{c^2}\,\phi_{pm},\qquad \phi_{pm}\phi_{pn} = \delta_{mn}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $D_{ac}$ | `second_difference(n, h, dev)` | (n, n) | 1D second derivative with the edges clamped ($u = 0$ outside) |
| $\Delta_{(ab)(cd)}$ | `kron(D, I) + kron(I, D)` | (n², n²) | pair index $p = (a, b)$ numbers the grid points: the 2D Laplacian |
| $\Delta_{pq}$ | `lap[inside][:, inside]` | (P, P) | keep only the points inside the drum (circle mask); the rim is clamped |
| $\omega_m,\ \phi_{pm}$ | `torch.linalg.eigh(-lap)` | (P,), (P, P) | eigenvalues $\omega^2/c^2$ and orthonormal mode shapes in the columns |
| $\phi_{pm}\phi_{pn}$ | `einsum('pm,pn->mn', phi, phi)` | (P, P) | repeated $p$ ⇒ summed: the identity matrix |

Each panel shows one mode $\phi_m(x, y)\cos(\omega_m t)$; white lines inside are nodal lines that never move. For a square of side $L$ the exact frequencies are $\omega_{mn} = \frac{c\pi}{L}\sqrt{m^2 + n^2}$, so the ratios to the fundamental are $\sqrt{(m^2+n^2)/2}$ = 1, 1.58, 1.58, 2, 2.24 …: not whole numbers, which is why a drum does not sound like a string. Pairs like $(1,2)$ and $(2,1)$ share a frequency, and eigh returns some mixture of them. For a circle the exact ratios come from zeros of Bessel functions (1, 1.59, 2.14, 2.30 …); the staircase rim of the grid shifts them by a few percent. Units: $c = 1$, square side or circle diameter $L = 1$; time is shown in fundamental periods.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/membrane_modes/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

# exact frequency ratios ω_m / ω_1 of the first six modes
SQUARE = [math.sqrt((m * m + n * n) / 2) for m, n in [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1)]]
BESSEL_ZEROS = [2.40483, 3.83171, 3.83171, 5.13562, 5.13562, 5.52008]  # j_{0,1}, j_{1,1}, j_{1,1}, j_{2,1}, …
CIRCLE = [j / BESSEL_ZEROS[0] for j in BESSEL_ZEROS]


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = params["n_grid"]
    X, Y, h = tensors.grid(n, dev)
    D = tensors.second_difference(n, h, dev)
    inside = tensors.drum_mask(X, Y, params["shape"])

    omega, shapes = solve.solve(D, inside, tensors.C)

    exact = SQUARE if params["shape"] == "square" else CIRCLE
    print(f"{int(inside.sum())} grid points inside the drum")
    for m in range(6):
        print(f"mode {m + 1}:  omega/omega_1 = {(omega[m] / omega[0]).item():.4f}   exact {exact[m]:.4f}")
    if params["shape"] == "square":
        print(f"fundamental omega_1 = {omega[0].item():.4f}   exact c*pi*sqrt(2)/L = {math.pi * math.sqrt(2):.4f}")
    else:
        print(f"fundamental omega_1 = {omega[0].item():.4f}   exact 2.40483 c/R = {2 * 2.40483:.4f}")
    flat = shapes.flatten(1)[:, inside].T  # (P, 6) back in the eigenvector layout
    error = (equation.overlap(flat) - torch.eye(6, dtype=flat.dtype, device=dev)).abs().max()
    print(f"orthonormality error max|phi^T phi - I| = {error:.1e}")
    return plot.animate(shapes, omega, inside, params["periods"], out_dir / "membrane_modes.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/membrane_modes/equation.py
r"""Physics → tensor translation.

A clamped membrane obeys the 2D wave equation ∂_t² u = c² ∇² u. Standing waves u = φ(x, y) cos(ωt) turn it
into an eigenvalue problem, −∇²φ = (ω/c)² φ.

On an n × n grid the point (a, b) gets the single index p = a·n + b, and the 2D Laplacian is a Kronecker sum
of the 1D second-difference matrix D:

    \Delta_{(ab)(cd)} = D_{ac} \delta_{bd} + \delta_{ac} D_{bd}        i.e.   Δ = D ⊗ I + I ⊗ D

The modes are the eigenvectors of −Δ (q summed):

    -\Delta_{pq} \phi_{qm} = (\omega_m / c)^2 \phi_{pm},        \phi_{pm} \phi_{pn} = \delta_{mn}    (p summed)
"""

import torch


def laplacian_2d(D: torch.Tensor) -> torch.Tensor:
    """Δ = D ⊗ I + I ⊗ D.   D: (n, n) → (n², n²)"""
    eye = torch.eye(D.shape[0], dtype=D.dtype, device=D.device)
    return torch.kron(D, eye) + torch.kron(eye, D)


def restrict(lap: torch.Tensor, inside: torch.Tensor) -> torch.Tensor:
    """Δ_{pq} for the points inside the drum only (outside points are clamped to 0).   inside: (n²,) bool → (P, P)"""
    return lap[inside][:, inside]


def modes(lap: torch.Tensor, c: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Angular frequencies ω_m (P,) lowest first and orthonormal mode shapes φ_{pm} (P, P) from eigh(−Δ)."""
    lam, phi = torch.linalg.eigh(-lap)  # eigh sorts the eigenvalues in ascending order
    return c * torch.sqrt(lam), phi


def overlap(phi: torch.Tensor) -> torch.Tensor:
    """φ_{pm} φ_{pn} — p summed. Orthonormal modes give the identity.   (P, M) → (M, M)"""
    return torch.einsum("pm,pn->mn", phi, phi)


def displacement(shapes: torch.Tensor, omega: torch.Tensor, t: float) -> torch.Tensor:
    """u_m(x, y, t) = φ_m(x, y) cos(ω_m t) for every shown mode.   shapes: (M, n, n), omega: (M,) → (M, n, n)"""
    return shapes * torch.cos(omega * t)[:, None, None]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/membrane_modes/tensors.py
"""Build the concrete tensors: the 1D stencil matrix, the grid and the drum mask."""

import torch

SIDE = 1.0  # square side or circle diameter L
C = 1.0  # wave speed on the membrane


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Interior points of [0, L]²: X, Y of shape (n, n) and the spacing h = L / (n + 1). The rim is not stored."""
    h = SIDE / (n + 1)
    s = torch.arange(1, n + 1, device=device, dtype=torch.float64) * h
    X, Y = torch.meshgrid(s, s, indexing="ij")
    return X, Y, h


def second_difference(n: int, h: float, device: torch.device) -> torch.Tensor:
    """D_{ac} of shape (n, n): (δ_{a+1,c} − 2δ_{ac} + δ_{a−1,c}) / h², the clamped rim built in."""
    main = torch.full((n,), -2.0, device=device, dtype=torch.float64)
    side = torch.ones(n - 1, device=device, dtype=torch.float64)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / h**2


def drum_mask(X: torch.Tensor, Y: torch.Tensor, shape: str) -> torch.Tensor:
    """Which grid points belong to the drum, flattened to (n²,) bool (p = a·n + b, the same order as torch.kron)."""
    if shape == "square":
        inside = torch.ones_like(X, dtype=torch.bool)
    else:
        r = SIDE / 2
        inside = (X - r) ** 2 + (Y - r) ** 2 < r**2
    return inside.flatten()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/membrane_modes/solve.py
"""Solve the eigenvalue problem and put the first few mode shapes back on the square grid."""

import torch

from .equation import laplacian_2d, modes, restrict


def solve(D: torch.Tensor, inside: torch.Tensor, c: float, n_modes: int = 6) -> tuple[torch.Tensor, torch.Tensor]:
    """Return ω (P,) for all modes and the first n_modes shapes on the grid: (n_modes, n, n), zero outside the drum."""
    n = D.shape[0]
    lap = restrict(laplacian_2d(D), inside)  # (P, P)
    omega, phi = modes(lap, c)
    shapes = torch.zeros(n_modes, n * n, dtype=phi.dtype, device=phi.device)
    shapes[:, inside] = phi[:, :n_modes].T  # scatter each mode back onto the full grid
    shapes = shapes.view(n_modes, n, n)
    # sign convention: make the largest swing of each mode positive (eigh may return −φ)
    peak = shapes.flatten(1).gather(1, shapes.flatten(1).abs().argmax(1, keepdim=True))
    return omega, shapes * torch.sign(peak)[:, :, None]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/membrane_modes/plot.py
"""Animate the first six modes side by side, each oscillating at its own frequency."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import displacement


def animate(shapes: torch.Tensor, omega: torch.Tensor, inside: torch.Tensor, periods: float, path: Path) -> Path:
    n_modes, n, _ = shapes.shape
    w = omega[:n_modes]
    T1 = 2 * math.pi / w[0].item()  # fundamental period
    times = np.linspace(0.0, periods * T1, 181)
    outside = ~inside.view(n, n).cpu().numpy()
    shapes = shapes / shapes.flatten(1).abs().max(1).values[:, None, None]  # each mode scaled to peak 1 for display
    limit = 1.0
    cmap = plt.get_cmap("RdBu_r", 16).with_extremes(bad="white")  # white = outside the drum

    fig, axes = plt.subplots(2, 3, figsize=(7, 4.6))
    images = []
    for m, ax in enumerate(axes.flat):
        frame = np.ma.array(shapes[m].cpu().numpy(), mask=outside)
        images.append(ax.imshow(frame.T, origin="lower", cmap=cmap, vmin=-limit, vmax=limit, extent=(0, 1, 0, 1)))
        ax.set_title(f"mode {m + 1}:  ω/ω₁ = {(w[m] / w[0]).item():.3f}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    title = fig.suptitle("membrane modes  φ(x, y) cos(ωt)    t = 0.00 T₁", fontsize=11)

    def update(step: int):
        u = displacement(shapes, w, times[step]).cpu().numpy()  # (M, n, n)
        for m, image in enumerate(images):
            image.set_data(np.ma.array(u[m], mask=outside).T)
        title.set_text(f"membrane modes  φ(x, y) cos(ωt)    t = {times[step] / T1:.2f} T₁")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(times), 90), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'shape': 'square',
    'n_grid': 30,
    'periods': 3.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.membrane_modes")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.membrane_modes.main")
problem_dir = Path("problems/waves/membrane_modes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
